# Assignment 3 — Data Preprocessing in the Right Order

**Dataset:** UCI Census Income / Adult dataset. UCI reports 48,842 instances, 14 features, mixed categorical/integer data, and missing values. The target is whether annual income exceeds $50K. citeturn1search6

**Source:** https://archive.ics.uci.edu/dataset/20/census+income

**Important:** The notebook splits first. All learned preprocessing parameters are fitted on training data only. Q9 intentionally demonstrates the forbidden leakage case.

In [ ]:
!pip -q install imbalanced-learn

In [ ]:
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import pairwise_distances
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

columns = ["age","workclass","fnlwgt","education","education-num","marital-status",
           "occupation","relationship","race","sex","capital-gain","capital-loss",
           "hours-per-week","native-country","income"]
url_train = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
url_test = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
train_raw = pd.read_csv(url_train, names=columns, skipinitialspace=True, na_values="?")
test_raw = pd.read_csv(url_test, names=columns, skipinitialspace=True, skiprows=1, na_values="?")
df = pd.concat([train_raw, test_raw], ignore_index=True)
df["income"] = df["income"].astype("string").str.replace(".", "", regex=False).str.strip()
print(df.shape)
print(df["income"].value_counts(normalize=True))

## Q1. Split first, stratified

In [ ]:
X = df.drop(columns="income")
y = df["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("train:", X_train.shape, "test:", X_test.shape)
print("Why first: preprocessing statistics must not learn from test data.")
print("A pre-split fit leaks information about the test distribution and makes evaluation optimistic.")

## Q2. Three missing-value strategies

In [ ]:
# Use capital-gain for a controlled numeric missingness comparison.
# The source already contains natural missing categorical values; this controlled mask lets
# us compare drop, median and KNN imputation on a numeric column without fitting on test data.
rng = np.random.default_rng(42)
X_train_q2 = X_train.copy()
mask = rng.random(len(X_train_q2)) < 0.05
X_train_q2.loc[mask, "capital-gain"] = np.nan

drop_result = X_train_q2["capital-gain"].dropna()
median_value = X_train_q2["capital-gain"].median()
median_result = X_train_q2["capital-gain"].fillna(median_value)

numeric_cols = ["age","fnlwgt","education-num","capital-gain","capital-loss","hours-per-week"]
knn = KNNImputer(n_neighbors=5)
knn_result = pd.Series(
    knn.fit_transform(X_train_q2[numeric_cols])[:, numeric_cols.index("capital-gain")],
    index=X_train_q2.index
)
comparison = pd.DataFrame({
    "method": ["drop rows","median","KNN"],
    "mean": [drop_result.mean(), median_result.mean(), knn_result.mean()],
    "std": [drop_result.std(), median_result.std(), knn_result.std()]
})
display(comparison)
print("Choice: median is the robust, simple default for this heavily zero-inflated/skewed monetary field; KNN is useful but more computationally expensive.")

## Q3. IQR, z-score and percentile capping

In [ ]:
s = X_train["capital-gain"].astype(float)
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
iqr_mask = (s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)

z_mask = (np.abs((s - s.mean()) / s.std(ddof=0)) > 3)

lo, hi = s.quantile([0.01, 0.99])
cap_mask = (s < lo) | (s > hi)

print("IQR flags:", int(iqr_mask.sum()))
print("z-score flags:", int(z_mask.sum()))
print("1st/99th percentile flags:", int(cap_mask.sum()))
print("IQR ∩ z-score:", int((iqr_mask & z_mask).sum()))
print("IQR ∩ percentile:", int((iqr_mask & cap_mask).sum()))
print("z-score ∩ percentile:", int((z_mask & cap_mask).sum()))
print("Choice: percentile capping is preferred here because capital-gain is extremely skewed and capping limits leverage without deleting rows.")

## Q4. One-hot, ordinal and frequency encoding

In [ ]:
low_card = X_train[["workclass"]].fillna("Missing")
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe_out = ohe.fit_transform(low_card)

ordered = X_train[["education"]].fillna("Missing")
education_order = ["Preschool","1st-4th","5th-6th","7th-8th","9th","10th","11th","12th",
                   "HS-grad","Some-college","Assoc-voc","Assoc-acdm","Bachelors","Masters","Prof-school","Doctorate","Missing"]
ordinal = OrdinalEncoder(categories=[education_order], handle_unknown="use_encoded_value", unknown_value=-1)
ord_out = ordinal.fit_transform(ordered)

high_card = X_train["native-country"].fillna("Missing")
freq_map = high_card.value_counts(normalize=True)
freq_out = high_card.map(freq_map).to_numpy().reshape(-1,1)

print("one-hot columns:", ohe_out.shape[1])
print("ordinal columns:", ord_out.shape[1])
print("frequency columns:", freq_out.shape[1])

## Q5. Unseen test category must not crash

In [ ]:
safe_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
safe_encoder.fit(X_train[["workclass"]].fillna("Missing"))
test_demo = X_test[["workclass"]].fillna("Missing").copy()
test_demo.iloc[0, 0] = "__UNSEEN_WORKCLASS__"
encoded = safe_encoder.transform(test_demo)
print("transformed shape:", encoded.shape)
print("unseen category handled without crash: PASS")

## Q6. Scale a skewed numeric column

In [ ]:
feature = X_train["capital-gain"].fillna(X_train["capital-gain"].median()).to_numpy().reshape(-1,1)
scaled_standard = StandardScaler().fit_transform(feature).ravel()
scaled_minmax = MinMaxScaler().fit_transform(feature).ravel()
scaled_robust = RobustScaler().fit_transform(feature).ravel()

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].hist(feature.ravel(), bins=40); ax[0].set_title("Original")
ax[1].hist(scaled_standard, bins=40); ax[1].set_title("StandardScaler")
ax[2].hist(scaled_minmax, bins=40); ax[2].set_title("MinMaxScaler")
ax[3].hist(scaled_robust, bins=40); ax[3].set_title("RobustScaler")
plt.tight_layout()
print("RobustScaler handles outliers best because its center and scale use median/IQR rather than mean/std.")

## Q7. Skewness before and after log1p and Yeo-Johnson

In [ ]:
from scipy.stats import skew
x = X_train["capital-gain"].fillna(0).to_numpy().reshape(-1,1)
log_x = np.log1p(x).ravel()
yj_x = PowerTransformer(method="yeo-johnson", standardize=False).fit_transform(x).ravel()
print("original skewness:", skew(x.ravel()))
print("log1p skewness:", skew(log_x))
print("Yeo-Johnson skewness:", skew(yj_x))

## Q8. ColumnTransformer inside a Pipeline

In [ ]:
numeric_cols = ["age","fnlwgt","education-num","capital-gain","capital-loss","hours-per-week"]
categorical_cols = ["workclass","education","marital-status","occupation","relationship","race","sex","native-country"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

pipeline = Pipeline([("preprocessor", preprocessor)])
Xtr = pipeline.fit_transform(X_train)
Xte = pipeline.transform(X_test)
print("train output:", Xtr.shape)
print("test output:", Xte.shape)
assert Xtr.shape[1] == Xte.shape[1]

## Q9. Demonstrate leakage

In [ ]:
target_col = "capital-gain"
full_scaler = StandardScaler()
full_scaled_test = full_scaler.fit_transform(
    df[[target_col]].fillna(df[target_col].median())
)[len(X_train):len(X_train)+5].ravel()

train_scaler = StandardScaler()
train_scaled_test = train_scaler.fit(
    X_train[[target_col]].fillna(X_train[target_col].median())
).transform(X_test[[target_col]].fillna(X_train[target_col].median()))[:5].ravel()

difference = full_scaled_test - train_scaled_test
print("leaky first five:", full_scaled_test)
print("correct first five:", train_scaled_test)
print("difference:", difference)
print("Leakage occurs because the full-data scaler learns test distribution statistics before evaluation. The resulting test values are no longer independent of the test set.")

## Q10. SMOTE on training set only

In [ ]:
print("class ratio:")
print(y_train.value_counts(normalize=True))
X_train_encoded = Xtr
X_test_encoded = Xte

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_encoded, y_train)
print("before:", y_train.value_counts().to_dict())
print("after: ", pd.Series(y_res).value_counts().to_dict())
print("The test set is untouched so its class distribution remains a realistic hold-out evaluation set.")

## Q11. Save fitted pipeline and transform one raw row

In [ ]:
# Save the fitted preprocessing pipeline.
joblib.dump(pipeline, "pipeline.joblib")

loaded_pipeline = joblib.load("pipeline.joblib")
new_row = X_test.iloc[[0]].copy()
new_transformed = loaded_pipeline.transform(new_row)
print("single raw row transformed shape:", new_transformed.shape)
assert new_transformed.shape[0] == 1

## Q12. Final choices
1. **Q2:** Median imputation is the preferred practical default for `capital-gain` because it is robust and cheap.
2. **Q3:** Percentile capping is preferred because it limits extreme leverage without deleting observations.
3. **Q6:** RobustScaler is preferred for an outlier-heavy skewed variable because it uses robust statistics.